## 1. ImagePSNR (Peak Signal-to-Noise Ratio)
**Measures:** How much noise is present between the reference and generated images based on the Mean Squared Error (MSE).

**Good for:** Sensitive to small pixel-wise differences.

**Limitations:** Does not account for visual structure.

**Scale:** Higher is better, measured in decibels (dB).

**Example:** A blurry but pixel-wise close image will score high.


## 2. ImageSSIM (Structural Similarity Index)
**Measures:** Structural, contrast, and luminance similarity.

**Good for:** More robust to lighting changes and better captures visual quality than ImagePSNR.

**Limitations:** May still miss subtle distortions.

**Scale:** 1 = perfect match; 0.5 = moderate; lower values = poor match.

**Example:** If an image is brightened but still sharp and structured, ImageSSIM remains high.


## 3. ImageAverageHash (aHash)
**Measures:** Visual pattern similarity using a binary hash of a downscaled grayscale image.

**Good for:** Detecting similar images, not pixel-wise comparison.

**Limitations:** Does not capture fine details or colors.

**Scale:** [0, 1], where 1 is identical.

**Example:** A resized or slightly shifted image still has high aHash.

## 4. ImageHistogramMatch
**Measures:** Color distribution similarity via RGB histograms.

**Good for:** Checking color similarity.

**Limitations:** Ignores structure and shape.

**Scale:** [0, 1], where 1 means identical histograms.

**Example:** Same layout with recolored image gives high histogram match but low ImagePSNR.

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter
import warnings
from typing import Any, Iterable, List
from PIL import Image
import pandas as pd

In [ ]:
class ImageMetric():
    """
    Abstract base class for metrics that operate on image data.
    Input can be various image representations (e.g., np.array, image path).
    """
    def __init__(self, **kwargs: Any):
        """
        Base initializer for metric classes.
        Can be used to set up models or configurations.
        """
        pass

    def _single_calculate(
        self, generated_item: Any, reference_item: Any, **kwargs: Any
    ) -> float | dict:
        """Abstract method for calculating a metric on a single pair of items."""
        raise NotImplementedError("This metric does not have a single calculation method.")

    def _batch_calculate(
        self,
        generated_items: Iterable | np.ndarray | pd.Series,
        reference_items: Iterable | np.ndarray | pd.Series,
        **kwargs: Any,
    ) -> List[float] | List[dict] | np.ndarray | pd.Series:
        """Abstract method for calculating a metric on a batch of items."""
        raise NotImplementedError("This metric does not have a batch calculation method.")



class ImageSSIM(ImageMetric):
    """
    Computes the Structural Similarity Index (ImageSSIM) between two images or batches.
    Pure NumPy + SciPy implementation. Output is normalized to [0, 1].
    """
    def __init__(self, resize: bool = True, **kwargs: Any):
        super().__init__(**kwargs)
        self.resize = resize

    def _compute_ssim_grayscale(self, img1: np.ndarray, img2: np.ndarray) -> float:
        K1, K2 = 0.01, 0.03
        L = 255.0
        C1 = (K1 * L) ** 2
        C2 = (K2 * L) ** 2

        mu1 = gaussian_filter(img1, sigma=1.5)
        mu2 = gaussian_filter(img2, sigma=1.5)

        mu1_sq = mu1 ** 2
        mu2_sq = mu2 ** 2
        mu1_mu2 = mu1 * mu2

        sigma1_sq = gaussian_filter(img1 * img1, sigma=1.5) - mu1_sq
        sigma2_sq = gaussian_filter(img2 * img2, sigma=1.5) - mu2_sq
        sigma12 = gaussian_filter(img1 * img2, sigma=1.5) - mu1_mu2

        numerator = (2 * mu1_mu2 + C1) * (2 * sigma12 + C2)
        denominator = (mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2)
        ssim_map = numerator / (denominator + 1e-8)
        return np.mean(ssim_map)

    def _single_calculate(self, generated_item: Any, reference_item: Any, **kwargs: Any) -> float:
        if isinstance(generated_item, Image.Image):
            generated_item = generated_item.convert("RGB")
            generated_item = np.array(generated_item)
        if isinstance(reference_item, Image.Image):
            reference_item = reference_item.convert("RGB")
            reference_item = np.array(reference_item)

        if generated_item.dtype != np.uint8:
            generated_item = (np.clip(generated_item, 0, 1) * 255).astype(np.uint8)
        if reference_item.dtype != np.uint8:
            reference_item = (np.clip(reference_item, 0, 1) * 255).astype(np.uint8)

        if generated_item.shape != reference_item.shape:
            if not self.resize:
                raise ValueError("Input images for ImageSSIM must have the same dimensions.")
            ref_img = Image.fromarray(reference_item)
            gen_img = Image.fromarray(generated_item).resize(ref_img.size, Image.Resampling.LANCZOS)
            generated_item = np.array(gen_img)
            reference_item = np.array(ref_img)

        # RGB channel-wise ImageSSIM, then normalized to [0,1]
        if generated_item.ndim == 3 and generated_item.shape[2] == 3:
            ssim_val = np.mean([
                self._compute_ssim_grayscale(generated_item[..., c], reference_item[..., c])
                for c in range(3)
            ])
        else:
            ssim_val = self._compute_ssim_grayscale(generated_item, reference_item)

        # Normalize to [0,1]
        return (ssim_val + 1) / 2

    def _batch_calculate(self, generated_items: Iterable, reference_items: Iterable, **kwargs: Any) -> List[float]:
        return [self._single_calculate(gen, ref, **kwargs) for gen, ref in zip(generated_items, reference_items)]



class ImagePSNR(ImageMetric):
    """
    Computes the Peak Signal-to-Noise Ratio (ImagePSNR) between two images or batches.
    Uses a pure NumPy implementation. Output is in decibels (dB).
    """
    def __init__(self, resize: bool = True, **kwargs: Any):
        super().__init__(**kwargs)
        self.resize = resize

    def _single_calculate(self, generated_item: Any, reference_item: Any, **kwargs: Any) -> float:
        if isinstance(generated_item, Image.Image):
            generated_item = generated_item.convert("RGB")
            generated_item = np.array(generated_item)
        if isinstance(reference_item, Image.Image):
            reference_item = reference_item.convert("RGB")
            reference_item = np.array(reference_item)

        if generated_item.dtype != np.uint8:
            generated_item = (np.clip(generated_item, 0, 1) * 255).astype(np.uint8)
        if reference_item.dtype != np.uint8:
            reference_item = (np.clip(reference_item, 0, 1) * 255).astype(np.uint8)

        if generated_item.shape != reference_item.shape:
            if not self.resize:
                raise ValueError("Input images for ImagePSNR must have the same dimensions.")
            ref_img = Image.fromarray(reference_item)
            gen_img = Image.fromarray(generated_item).resize(ref_img.size, Image.Resampling.LANCZOS)
            generated_item = np.array(gen_img)
            reference_item = np.array(ref_img)

        mse = np.mean((generated_item.astype(np.float32) - reference_item.astype(np.float32)) ** 2)
        if mse == 0:
            return float("inf")
        return 10 * np.log10((255.0 ** 2) / mse)

    def _batch_calculate(self, generated_items: Iterable, reference_items: Iterable, **kwargs: Any) -> List[float]:
        return [self._single_calculate(gen, ref, **kwargs) for gen, ref in zip(generated_items, reference_items)]





class ImageAverageHash(ImageMetric):
    """
    Normalized average hash (aHash) similarity for images.
    This method compares 8x8 grayscale downsampled images and computes
    Hamming similarity of the resulting binary hash. The score is normalized to [0, 1].
    """
    def __init__(self, **kwargs: Any):
        """Initialize the aHash-based image similarity metric."""
        super().__init__(**kwargs)

    def _single_calculate(
        self, generated_item: Any, reference_item: Any, **kwargs: Any
    ) -> float | dict:
        """
        Calculate normalized aHash similarity for a single pair of images.
        """
        if isinstance(generated_item, np.ndarray):
            if generated_item.dtype != np.uint8:
                 generated_item = (np.clip(generated_item, 0, 1) * 255).astype(np.uint8)
            generated_item = Image.fromarray(generated_item)
        if isinstance(reference_item, np.ndarray):
            if reference_item.dtype != np.uint8:
                 reference_item = (np.clip(reference_item, 0, 1) * 255).astype(np.uint8)
            reference_item = Image.fromarray(reference_item)

        gen_resized = generated_item.convert("L").resize((8, 8), Image.Resampling.LANCZOS)
        ref_resized = reference_item.convert("L").resize((8, 8), Image.Resampling.LANCZOS)
        gen_array = np.array(gen_resized, dtype=np.float32)
        ref_array = np.array(ref_resized, dtype=np.float32)
        gen_mean = gen_array.mean()
        ref_mean = ref_array.mean()
        gen_hash = (gen_array > gen_mean).astype(np.uint8).flatten()
        ref_hash = (ref_array > ref_mean).astype(np.uint8).flatten()
        similarity = 1.0 - np.sum(gen_hash != ref_hash) / len(gen_hash)
        return similarity

    def _batch_calculate(
        self, generated_items: Iterable, reference_items: Iterable, **kwargs: Any
    ) -> List[float]:
        """
        Calculate normalized aHash similarity for a batch of image pairs.
        """
        results = []
        for gen, ref in zip(generated_items, reference_items):
            results.append(self._single_calculate(gen, ref, **kwargs))
        return results


class ImageHistogramMatch(ImageMetric):
    """
    Color histogram-based similarity metric for images.
    Computes normalized histogram intersection between RGB histograms of two images.
    The output is a similarity score in the range [0, 1], where 1 means the histograms are identical.
    """
    def __init__(self, **kwargs: Any):
        """Initialize the histogram-based similarity metric."""
        super().__init__(**kwargs)

    def _single_calculate(
        self, generated_item: Any, reference_item: Any, **kwargs: Any
    ) -> float | dict:
        """
        Calculate histogram intersection similarity for a single pair of images.
        """
        if isinstance(generated_item, np.ndarray):
            if generated_item.dtype != np.uint8:
                 generated_item = (np.clip(generated_item, 0, 1) * 255).astype(np.uint8)
            generated_item = Image.fromarray(generated_item)
        if isinstance(reference_item, np.ndarray):
            if reference_item.dtype != np.uint8:
                 reference_item = (np.clip(reference_item, 0, 1) * 255).astype(np.uint8)
            reference_item = Image.fromarray(reference_item)

        bins = kwargs.get("bins", 256)
        gen_arr = np.array(generated_item.convert("RGB"))
        ref_arr = np.array(reference_item.convert("RGB"))
        intersection = 0.0
        total = 0.0
        for ch in range(3):
            gen_hist = np.histogram(gen_arr[:, :, ch], bins=bins, range=(0, 255))[0]
            ref_hist = np.histogram(ref_arr[:, :, ch], bins=bins, range=(0, 255))[0]
            intersection += np.sum(np.minimum(gen_hist, ref_hist))
            total += np.sum(gen_hist)
        similarity = intersection / total if total > 0 else 0.0
        return similarity

    def _batch_calculate(
        self, generated_items: Iterable, reference_items: Iterable, **kwargs: Any
    ) -> List[float]:
        """
        Calculate histogram intersection similarity for a batch of image pairs.
        """
        results = []
        for gen, ref in zip(generated_items, reference_items):
            results.append(self._single_calculate(gen, ref, **kwargs))
        return results

NameError: name 'Any' is not defined

In [ ]:
import os
image_folder = "./data/images"
query_path = "./data/images/dog.jpg"

image_paths = [
    os.path.join(image_folder, fname)
    for fname in os.listdir(image_folder)
    if fname.lower().endswith((".jpg", ".png")) and fname != os.path.basename(query_path)
]

query_image = Image.open(query_path)
target_images = [Image.open(path) for path in image_paths]

query_batch = [query_image] * len(target_images)

ahash_metric = ImageAverageHash()
hist_metric = ImageHistogramMatch()
ss_metric = ImageSSIM()
ps_metric = ImagePSNR()

ahash_scores = ahash_metric._batch_calculate(query_batch, target_images)
hist_scores = hist_metric._batch_calculate(query_batch, target_images)
ssim_scores = ss_metric._batch_calculate(query_batch, target_images)
psnr_scores = ps_metric._batch_calculate(query_batch, target_images)

for path, ahash, hist, ssim, ps in zip(image_paths, ahash_scores, hist_scores, ssim_scores, psnr_scores):
    print(f"{os.path.basename(query_path)} vs {os.path.basename(path)}\n")
    print(f"  ImageAverageHash Similarity:   {ahash:.4f}\n")
    print(f"  ImageHistogramMatch Similarity: {hist:.4f}\n")
    print(f"  ImageSSIM Similarity: {ssim:.4f}\n")
    print(f"  ImagePSNR Similarity: {ps:.4f}\n")